---
## 1. Configuration et Imports

In [1]:
# Imports nécessaires
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Chemins
BASE_PATH = '/home/henintsoa/CFIM'
FINAL_PATH = os.path.join(BASE_PATH, 'final/data')

print(" Configuration chargée")
print(f" Chemin des données: {FINAL_PATH}")

 Configuration chargée
 Chemin des données: /home/henintsoa/CFIM/final/data


---
## 2. Chargement des Données

In [2]:
# Charger les données météo standardisées
print(" CHARGEMENT DES DONNÉES MÉTÉO")
print("=" * 60)

df_meteo = pd.read_csv(os.path.join(FINAL_PATH, 'meteo_standardise.csv'))

print(f"\n Dimensions: {df_meteo.shape}")
print(f" Colonnes: {list(df_meteo.columns)}")

# Parser la date
df_meteo['date_parsed'] = pd.to_datetime(df_meteo['date'], format='%d/%m/%Y', errors='coerce')

print(f"\n Période: {df_meteo['date_parsed'].min().date()} à {df_meteo['date_parsed'].max().date()}")
print(f" Zones uniques: {df_meteo['zone'].nunique()}")

display(df_meteo.head())

 CHARGEMENT DES DONNÉES MÉTÉO

 Dimensions: (1346, 23)
 Colonnes: ['date', 'zone', 'vent', 'etat_mer', 'temps', 'vent_vitesse_min', 'vent_vitesse_max', 'vent_rafales', 'vent_direction', 'vent_direction_deg', 'mer_score_min', 'mer_score_max', 'mer_hauteur_min', 'mer_hauteur_max', 'temps_precipitation', 'temps_orage', 'temps_visibilite_reduite', 'temps_clair', 'temps_nuageux', 'temps_intensite', 'temps_score_danger', 'score_risque', 'categorie_risque']

 Période: 2019-09-06 à 2022-12-31
 Zones uniques: 67


,date,zone,vent,etat_mer,temps,vent_vitesse_min,vent_vitesse_max,vent_rafales,vent_direction,vent_direction_deg,...,temps_precipitation,temps_orage,temps_visibilite_reduite,temps_clair,temps_nuageux,temps_intensite,temps_score_danger,score_risque,categorie_risque,date_parsed
0,06/09/2019,CAP D'AMBRE A MAHANORO,10/15 kt atteignant 20/25 kt au nord d'Antalaha,agitée à forte,Pluies faible a modérée,10.0,15.0,25.0,E,90.0,...,1,0,0,0,0,2,3,49,Élevé (40-60),2019-09-06
1,06/09/2019,MAHANORO AU CAP SAINTE MARIE,05/10 kt devenant progressivement secteur sud ...,agitée à forte,pluies,5.0,10.0,30.0,E,90.0,...,1,0,0,0,0,0,1,43,Élevé (40-60),2019-09-06
2,06/09/2019,CAP D'AMBRE A BESALAMPY,"15/20 kt localement 20 kt au sud de Majunga, v...",Non spécifié,Temps sec,15.0,20.0,20.0,E,90.0,...,0,0,0,1,0,0,0,15,Faible (0-20),2019-09-06
3,06/09/2019,BESALAMPY A MOROMBE,15/20 kt,"agitée a forte, très forte près Morombe dans l...",Temps sec,15.0,20.0,NaN,NaN,NaN,...,0,0,0,1,0,0,0,45,Élevé (40-60),2019-09-06
4,06/09/2019,MOROMBE A CAP SAINTE MARIE,20/25 kt atteignant 30/35 kt entre Morombe et,Non spécifié,Temps partiellement nuageux,20.0,25.0,35.0,E,90.0,...,0,0,0,0,1,0,0,35,Modéré (20-40),2019-09-06


In [3]:
# Charger les données d'incidents - VERSION CORRIGÉE
print(" CHARGEMENT DES DONNÉES D'INCIDENTS")
print("=" * 60)

# TOUJOURS charger les données brutes pour avoir TOUS les incidents
df_incidents_raw = pd.read_csv(os.path.join(BASE_PATH, 'csv', 'incidents_maritimes_complets_2017_2022.csv'))

print(f"\n Données brutes: {len(df_incidents_raw)} lignes")

# Filtrer les VRAIS incidents (exclure "Aucun incident maritime")
df_incidents = df_incidents_raw[
    ~df_incidents_raw['description'].str.contains('Aucun incident', na=False, case=False)
].copy()

print(f" VRAIS incidents identifiés: {len(df_incidents)}")

# Parser la date
df_incidents['date_incident'] = pd.to_datetime(df_incidents['Date debut'], format='%d/%m/%Y', errors='coerce')

# Renommer les colonnes pour cohérence
df_incidents.rename(columns={
    'Types': 'type_incident',
    'Region': 'region',
    'Longitude': 'longitude',
    'Latitude': 'latitude'
}, inplace=True)

# Vérifier les coordonnées disponibles
has_coords = df_incidents['longitude'].notna() & df_incidents['latitude'].notna()
print(f"\n Incidents avec coordonnées GPS: {has_coords.sum()} / {len(df_incidents)}")

# Aperçu
print(f"\n Colonnes: {list(df_incidents.columns)}")
print(f" Période: {df_incidents['date_incident'].min().date()} à {df_incidents['date_incident'].max().date()}")

if len(df_incidents) > 0:
    print(f"\n Types d'incidents (top 5):")
    print(df_incidents['type_incident'].value_counts().head())

display(df_incidents[['Date debut', 'type_incident', 'region', 'District', 'Commune', 'longitude', 'latitude']].head(10))


 CHARGEMENT DES DONNÉES D'INCIDENTS

 Données brutes: 2215 lignes
 VRAIS incidents identifiés: 269

 Incidents avec coordonnées GPS: 263 / 269

 Colonnes: ['Date debut', 'Date fin', 'Thématique', 'Objets', 'District', 'Commune', 'Localite', 'region', 'type_incident', 'longitude', 'latitude', 'Personne concerne', 'Mort', 'Colonne1', 'description', 'Commentaire', 'Date_debut_dt', 'date_incident']
 Période: 2017-02-05 à 2022-12-31

 Types d'incidents (top 5):
type_incident
SAR             13
ACC_NAUFRAGE    10
ACC_NOYADE       8
ACC_PANNE        4
ACC_ECHOUAGE     4
Name: count, dtype: int64


,Date debut,type_incident,region,District,Commune,longitude,latitude
35,05/02/2017,NaN,NaN,MADAGASCAR,NaN,46.325890,-15.766938
55,25/02/2017,NaN,NaN,Katsepy Mahajanga/ Madagascar,NaN,46.289816,-15.737907
65,07/03/2017,NaN,NaN,OCEAN INDIEN,NaN,54.740555,-21.401068
75,17/03/2017,NaN,NaN,madagascar,NaN,43.872715,-18.042964
89,31/03/2017,NaN,NaN,OCEAN INDIEN,NaN,68.761516,-20.827302
106,17/04/2017,NaN,NaN,MADAGASCAR,NaN,49.420532,-18.150623
109,20/04/2017,NaN,NaN,MADAGASCAR,NaN,50.063335,-13.360009
128,09/05/2017,NaN,NaN,MADAGASCAR,NaN,49.869580,-15.966218
189,09/07/2017,NaN,NaN,OCEAN INDIEN,NaN,59.665862,13.417318
191,11/07/2017,NaN,NaN,MADAGASCAR,NaN,49.502451,-17.514099


In [4]:
# Charger les données satellites (optionnel)
print(" CHARGEMENT DES DONNÉES SATELLITES (OPTIONNEL)")
print("=" * 60)

swh_file = os.path.join(FINAL_PATH, 'donnees_satellites_swh.csv')

if os.path.exists(swh_file):
    df_swh = pd.read_csv(swh_file)
    df_swh['date'] = pd.to_datetime(df_swh['date'])
    print(f"\n Données satellites chargées")
    print(f"   Période: {df_swh['date'].min().date()} à {df_swh['date'].max().date()}")
    print(f"   Enregistrements: {len(df_swh)}")
    HAS_SATELLITE = True
else:
    print("\n Fichier satellites non trouvé (notebook 2.1 non exécuté?)")
    print("   Continuons sans les données satellites")
    df_swh = None
    HAS_SATELLITE = False

 CHARGEMENT DES DONNÉES SATELLITES (OPTIONNEL)

 Données satellites chargées
   Période: 2017-01-02 à 2018-12-02
   Enregistrements: 288


---
## 3. Analyse de la Correspondance Temporelle

Avant de fusionner, analysons le chevauchement temporel entre les sources de données.

In [5]:
print(" ANALYSE DE LA CORRESPONDANCE TEMPORELLE")
print("=" * 60)

# Période météo
meteo_min = df_meteo['date_parsed'].min()
meteo_max = df_meteo['date_parsed'].max()

# Période incidents
incidents_min = df_incidents['date_incident'].min()
incidents_max = df_incidents['date_incident'].max()

print(f"\n Météo: {meteo_min.date()} → {meteo_max.date()}")
print(f" Incidents: {incidents_min.date()} → {incidents_max.date()}")

# Période de chevauchement
overlap_start = max(meteo_min, incidents_min)
overlap_end = min(meteo_max, incidents_max)

if overlap_start < overlap_end:
    overlap_days = (overlap_end - overlap_start).days
    print(f"\n Période commune: {overlap_start.date()} → {overlap_end.date()}")
    print(f"   Durée: {overlap_days} jours (~{overlap_days//30} mois)")
else:
    print("\n ATTENTION: Pas de période commune!")
    print("   Nous allons utiliser les données disponibles")
    overlap_start = meteo_min
    overlap_end = meteo_max

 ANALYSE DE LA CORRESPONDANCE TEMPORELLE

 Météo: 2019-09-06 → 2022-12-31
 Incidents: 2017-02-05 → 2022-12-31

 Période commune: 2019-09-06 → 2022-12-31
   Durée: 1212 jours (~40 mois)


In [6]:
# Filtrer les incidents dans la période commune
df_incidents_overlap = df_incidents[
    (df_incidents['date_incident'] >= overlap_start) &
    (df_incidents['date_incident'] <= overlap_end)
].copy()

print(f" INCIDENTS DANS LA PÉRIODE COMMUNE")
print("=" * 60)
print(f"\nNombre total d'incidents: {len(df_incidents_overlap)}")
print(f"Incidents avec coordonnées: {(df_incidents_overlap['longitude'].notna()).sum()}")
print(f"Incidents avec région: {(df_incidents_overlap['region'].notna()).sum()}")
print(f"Incidents avec date valide: {(df_incidents_overlap['date_incident'].notna()).sum()}")

if len(df_incidents_overlap) > 0:
    print(f"\n Types d'incidents (top 10):")
    print(df_incidents_overlap['type_incident'].value_counts().head(10))
    
    print(f"\n Distribution par année:")
    print(df_incidents_overlap['date_incident'].dt.year.value_counts().sort_index())


 INCIDENTS DANS LA PÉRIODE COMMUNE

Nombre total d'incidents: 189
Incidents avec coordonnées: 186
Incidents avec région: 20
Incidents avec date valide: 189

 Types d'incidents (top 10):
type_incident
SAR                13
ACC_NAUFRAGE       10
ACC_NOYADE          8
ACC_PANNE           4
ACC_ECHOUAGE        4
ACCIDENT_NOYADE     3
ACCIDENTS           3
ACC_INCENDIE        3
ACC_CHAVIREMENT     3
ACC                 2
Name: count, dtype: int64

 Distribution par année:
date_incident
2019    14
2020    49
2021    63
2022    63
Name: count, dtype: int64


---
## 4. Normalisation des Noms de Zones

Les noms de zones peuvent varier légèrement entre les sources. Nous les normalisons pour permettre la fusion.

In [7]:
import re
import json

def normaliser_zone(zone):
    """
    Normalise le nom d'une zone côtière pour la fusion.
    """
    if pd.isna(zone):
        return None
    
    zone = str(zone).upper().strip()
    
    # Supprimer les préfixes de type "PRÉVISION POUR LES COTES DE MADAGASCAR"
    zone = re.sub(r'^PR.*MADAGASCAR\s*', '', zone)
    zone = re.sub(r'^[A-Z]*\n\s*', '', zone)
    
    # Supprimer les caractères spéciaux et normaliser les espaces
    zone = zone.replace('\n', ' ').replace('\r', '')
    zone = ' '.join(zone.split())
    
    # Standardiser les noms communs
    replacements = {
        "CAP D AMBRE": "CAP D'AMBRE",
        "CAP DAMBRE": "CAP D'AMBRE",
        "SAINTE MARIE": "SAINTE-MARIE",
        "STE MARIE": "SAINTE-MARIE",
        "TOLIARY": "TOLIARA",
        "CAP ST ANDRE": "CAP SAINT ANDRE",
        "CAP ST MARIE": "CAP SAINTE MARIE",
    }
    
    for old, new in replacements.items():
        zone = zone.replace(old, new)
    
    return zone

# Charger le mapping zones-régions
mapping_file = os.path.join(FINAL_PATH, 'mapping_zones_regions.json')

if os.path.exists(mapping_file):
    with open(mapping_file, 'r', encoding='utf-8') as f:
        mapping_zones = json.load(f)
    print(f" Mapping zones-régions chargé: {len(mapping_zones)} zones")
else:
    print(" Fichier mapping_zones_regions.json non trouvé")
    print("   Création d'un mapping basique...")
    mapping_zones = {}

# Assigner une zone côtière à chaque incident basé sur la région
def assigner_zone_cotiere(row):
    """
    Assigne une zone côtière à un incident basé sur sa région ou ses coordonnées.
    """
    # Essayer d'abord par région
    if pd.notna(row['region']):
        region_norm = str(row['region']).upper().strip()
        # Trouver la zone correspondante dans le mapping
        for zone, data in mapping_zones.items():
            if isinstance(data, dict) and 'regions' in data:
                if region_norm in [r.upper() for r in data['regions']]:
                    return zone
    
    # Si pas de match par région, essayer par coordonnées
    if pd.notna(row['longitude']) and pd.notna(row['latitude']):
        lon, lat = row['longitude'], row['latitude']
        for zone, data in mapping_zones.items():
            if isinstance(data, dict) and all(k in data for k in ['lat_min', 'lat_max', 'lon_min', 'lon_max']):
                if (data['lat_min'] <= lat <= data['lat_max'] and 
                    data['lon_min'] <= lon <= data['lon_max']):
                    return zone
    
    return None

# Assigner les zones aux incidents
df_incidents_overlap['zone_cotiere'] = df_incidents_overlap.apply(assigner_zone_cotiere, axis=1)

print(f"\n ATTRIBUTION DES ZONES CÔTIÈRES AUX INCIDENTS")
print("=" * 60)
print(f"Incidents avec zone assignée: {df_incidents_overlap['zone_cotiere'].notna().sum()} / {len(df_incidents_overlap)}")
print(f"Incidents SANS zone: {df_incidents_overlap['zone_cotiere'].isna().sum()}")

# Pour les incidents sans zone, on utilise une approche nationale
# On va créer une variable incident_national qui sera 1 si n'importe quel incident s'est produit ce jour
print(f"\n Stratégie: Les incidents sans zone seront marqués au niveau NATIONAL")


 Mapping zones-régions chargé: 3 zones

 ATTRIBUTION DES ZONES CÔTIÈRES AUX INCIDENTS
Incidents avec zone assignée: 0 / 189
Incidents SANS zone: 189

 Stratégie: Les incidents sans zone seront marqués au niveau NATIONAL


---
## 5. Création du Dataset - NOUVELLE APPROCHE

**Problème identifié**: Les zones météo et les zones incidents ne correspondent pas bien (seulement ~5% de correspondances exactes).

**Solution**: Utiliser une approche basée sur la **DATE** comme clé principale:
- Pour chaque DATE dans la période, on agrège les conditions météo de toutes les zones
- On marque 1 si au moins un incident s'est produit ce jour-là (quelle que soit la zone)
- Cela permet de garder TOUS les incidents au lieu de seulement 6

In [8]:
print(" CRÉATION DU DATASET: APPROCHE PAR ZONE CÔTIÈRE")
print("=" * 60)

# Normaliser les zones météo
df_meteo['zone_norm'] = df_meteo['zone'].apply(normaliser_zone)

print(f"\n Données météo:")
print(f"   Total: {len(df_meteo)} observations")
print(f"   Zones uniques: {df_meteo['zone_norm'].nunique()}")
print(f"   Dates: {df_meteo['date_parsed'].min().date()} → {df_meteo['date_parsed'].max().date()}")

# Colonnes météo numériques à garder
colonnes_meteo_num = [
    'vent_vitesse_min', 'vent_vitesse_max', 'vent_rafales', 'vent_direction_deg',
    'mer_score_min', 'mer_score_max', 'mer_hauteur_min', 'mer_hauteur_max',
    'temps_precipitation', 'temps_orage', 'temps_visibilite_reduite',
    'temps_clair', 'temps_nuageux', 'temps_score_danger', 'score_risque'
]

# Vérifier les colonnes disponibles
colonnes_disponibles = [c for c in colonnes_meteo_num if c in df_meteo.columns]
print(f"\n Colonnes météo disponibles: {len(colonnes_disponibles)}")

# Préparer le dataset de base (Date x Zone)
df_meteo_prep = df_meteo[['date_parsed', 'zone_norm'] + colonnes_disponibles].copy()
df_meteo_prep = df_meteo_prep.dropna(subset=['zone_norm'])
df_meteo_prep.columns = ['date', 'zone'] + colonnes_disponibles

print(f"\n Dataset météo préparé: {len(df_meteo_prep)} observations (Date x Zone)")


 CRÉATION DU DATASET: APPROCHE PAR ZONE CÔTIÈRE

 Données météo:
   Total: 1346 observations
   Zones uniques: 55
   Dates: 2019-09-06 → 2022-12-31

 Colonnes météo disponibles: 15

 Dataset météo préparé: 1346 observations (Date x Zone)


---
## 6. Création de la Variable Cible - PAR DATE

Pour chaque jour, on marque 1 si au moins un incident s'est produit ce jour-là.

In [9]:
print(" CRÉATION DE LA VARIABLE CIBLE - APPROCHE HYBRIDE")
print("=" * 60)

# STRATÉGIE 1: Incidents par zone (quand on connaît la zone)
df_incidents_overlap['date_only'] = df_incidents_overlap['date_incident'].dt.date

# Créer un dictionnaire des incidents par (Date, Zone)
incidents_par_zone = {}
for _, row in df_incidents_overlap.iterrows():
    if pd.notna(row['date_only']) and pd.notna(row['zone_cotiere']):
        key = (pd.Timestamp(row['date_only']), row['zone_cotiere'])
        if key not in incidents_par_zone:
            incidents_par_zone[key] = []
        incidents_par_zone[key].append(row['type_incident'])

print(f"\n Incidents avec zone identifiée: {len(incidents_par_zone)} couples (Date, Zone)")

# STRATÉGIE 2: Incidents au niveau national (pour ceux sans zone)
incidents_nationaux = set()
for _, row in df_incidents_overlap.iterrows():
    if pd.notna(row['date_only']):
        incidents_nationaux.add(pd.Timestamp(row['date_only']))

print(f" Dates avec au moins un incident (niveau national): {len(incidents_nationaux)}")

# Total d'incidents uniques par date
incidents_par_date = df_incidents_overlap.groupby('date_only').size()
print(f"\n Distribution des incidents par date:")
print(f"   Dates avec 1 incident: {(incidents_par_date == 1).sum()}")
print(f"   Dates avec 2+ incidents: {(incidents_par_date >= 2).sum()}")
print(f"   Maximum incidents en 1 jour: {incidents_par_date.max()}")

# Assigner la variable cible au dataset météo
def determiner_incident(row):
    """
    Retourne 1 si un incident s'est produit ce jour dans cette zone OU à Madagascar ce jour-là.
    Approche hybride pour ne perdre AUCUN incident.
    """
    date = row['date']
    zone = row['zone']
    
    # Priorité 1: Incident spécifique à cette zone
    if (date, zone) in incidents_par_zone:
        return 1
    
    # Priorité 2: Incident national ce jour (quelque soit la zone)
    # On marque toutes les zones comme 1 si un incident s'est produit quelque part à Madagascar
    if date in incidents_nationaux:
        return 1
    
    return 0

df_meteo_prep['incident'] = df_meteo_prep.apply(determiner_incident, axis=1)

print(f"\n Distribution de la variable cible:")
print(df_meteo_prep['incident'].value_counts())
print(f"\nTaux d'incidents: {100*df_meteo_prep['incident'].mean():.2f}%")

# Vérification: combien de jours d'incidents capturés?
jours_incidents_captes = df_meteo_prep[df_meteo_prep['incident'] == 1]['date'].nunique()
print(f"\n Jours avec incident capturés: {jours_incidents_captes} / {len(incidents_nationaux)}")
print(f"   Taux de couverture: {100*jours_incidents_captes/len(incidents_nationaux):.1f}%")

# Si le taux de couverture est < 100%, afficher les dates manquantes
if jours_incidents_captes < len(incidents_nationaux):
    dates_captes = set(df_meteo_prep[df_meteo_prep['incident'] == 1]['date'].dt.date)
    dates_manquantes = incidents_nationaux - set([pd.Timestamp(d) for d in dates_captes])
    print(f"\n {len(dates_manquantes)} dates d'incidents non capturées (pas de données météo ces jours-là)")
    if len(dates_manquantes) <= 10:
        print(f"   Dates: {sorted([d.date() for d in dates_manquantes])}")


 CRÉATION DE LA VARIABLE CIBLE - APPROCHE HYBRIDE

 Incidents avec zone identifiée: 0 couples (Date, Zone)
 Dates avec au moins un incident (niveau national): 167

 Distribution des incidents par date:
   Dates avec 1 incident: 147
   Dates avec 2+ incidents: 20
   Maximum incidents en 1 jour: 3

 Distribution de la variable cible:
incident
0    1178
1     168
Name: count, dtype: int64

Taux d'incidents: 12.48%

 Jours avec incident capturés: 75 / 167
   Taux de couverture: 44.9%

 92 dates d'incidents non capturées (pas de données météo ces jours-là)


---
## 7. Ajout des Variables Temporelles

In [10]:
print(" AJOUT DES VARIABLES TEMPORELLES ET GÉOGRAPHIQUES")
print("=" * 60)

df_final = df_meteo_prep.copy()

# Variables temporelles
df_final['annee'] = df_final['date'].dt.year
df_final['mois'] = df_final['date'].dt.month
df_final['jour'] = df_final['date'].dt.day
df_final['jour_semaine'] = df_final['date'].dt.dayofweek  # 0=Lundi, 6=Dimanche
df_final['jour_annee'] = df_final['date'].dt.dayofyear

# Saison cyclonique (novembre à avril) - Période à risque à Madagascar
df_final['saison_cyclonique'] = df_final['mois'].isin([11, 12, 1, 2, 3, 4]).astype(int)

# Week-end
df_final['weekend'] = df_final['jour_semaine'].isin([5, 6]).astype(int)

# Variables cycliques pour le mois (pour capturer la saisonnalité)
df_final['mois_sin'] = np.sin(2 * np.pi * df_final['mois'] / 12)
df_final['mois_cos'] = np.cos(2 * np.pi * df_final['mois'] / 12)

# Variables cycliques pour le jour de l'année
df_final['jour_annee_sin'] = np.sin(2 * np.pi * df_final['jour_annee'] / 365)
df_final['jour_annee_cos'] = np.cos(2 * np.pi * df_final['jour_annee'] / 365)

# Encodage de la zone (One-Hot Encoding)
df_final = pd.get_dummies(df_final, columns=['zone'], prefix='zone', drop_first=False)

print("\n Variables ajoutées:")
print(f"   • Temporelles: année, mois, jour, jour_semaine, jour_année")
print(f"   • Saisonnières: saison_cyclonique, variables cycliques (sin/cos)")
print(f"   • Géographiques: encodage one-hot des zones côtières")
print(f"   • Dimensions totales: {df_final.shape}")

# Afficher les zones encodées
zones_cols = [c for c in df_final.columns if c.startswith('zone_')]
print(f"\n Zones encodées ({len(zones_cols)}):")
for i, z in enumerate(zones_cols[:10], 1):
    print(f"   {i}. {z}")
if len(zones_cols) > 10:
    print(f"   ... et {len(zones_cols)-10} autres")


 AJOUT DES VARIABLES TEMPORELLES ET GÉOGRAPHIQUES

 Variables ajoutées:
   • Temporelles: année, mois, jour, jour_semaine, jour_année
   • Saisonnières: saison_cyclonique, variables cycliques (sin/cos)
   • Géographiques: encodage one-hot des zones côtières
   • Dimensions totales: (1346, 83)

 Zones encodées (55):
   1. zone_AMPANIHY A TAOLAGNARO
   2. zone_ANTALAHA A FARAFANGANA
   3. zone_ANTALAHA A MAHANORO
   4. zone_ANTALAHA A MANANJARY
   5. zone_ANTALAHA A TAOLAGNARO
   6. zone_ANTALAHA A TOAMASINA
   7. zone_ANTALAHA AU CAP SAINTE-MARIE
   8. zone_AVIS DE GRAND FRAIS ASSOCIE A LA DEPRESSION TROPICALE ENTRE TOAMASINA
   9. zone_AVIS DE GRAND FRAIS ASSOCIE A LA DEPRESSION TROPICALE ENTRE TOAMASINA ET VATOMANDRY
   10. zone_BELNA' BESALAMPY A MOROMBE
   ... et 45 autres


In [11]:
# Aperçu du dataset
print(" APERÇU DU DATASET")
print("=" * 60)

print(f"\nDimensions: {df_final.shape}")
print(f"\nColonnes:")
for i, col in enumerate(df_final.columns):
    print(f"  {i+1}. {col}")

display(df_final.head())

 APERÇU DU DATASET

Dimensions: (1346, 83)

Colonnes:
  1. date
  2. vent_vitesse_min
  3. vent_vitesse_max
  4. vent_rafales
  5. vent_direction_deg
  6. mer_score_min
  7. mer_score_max
  8. mer_hauteur_min
  9. mer_hauteur_max
  10. temps_precipitation
  11. temps_orage
  12. temps_visibilite_reduite
  13. temps_clair
  14. temps_nuageux
  15. temps_score_danger
  16. score_risque
  17. incident
  18. annee
  19. mois
  20. jour
  21. jour_semaine
  22. jour_annee
  23. saison_cyclonique
  24. weekend
  25. mois_sin
  26. mois_cos
  27. jour_annee_sin
  28. jour_annee_cos
  29. zone_AMPANIHY A TAOLAGNARO
  30. zone_ANTALAHA A FARAFANGANA
  31. zone_ANTALAHA A MAHANORO
  32. zone_ANTALAHA A MANANJARY
  33. zone_ANTALAHA A TAOLAGNARO
  34. zone_ANTALAHA A TOAMASINA
  35. zone_ANTALAHA AU CAP SAINTE-MARIE
  36. zone_AVIS DE GRAND FRAIS ASSOCIE A LA DEPRESSION TROPICALE ENTRE TOAMASINA
  37. zone_AVIS DE GRAND FRAIS ASSOCIE A LA DEPRESSION TROPICALE ENTRE TOAMASINA ET VATOMANDRY
  38. z

,date,vent_vitesse_min,vent_vitesse_max,vent_rafales,vent_direction_deg,mer_score_min,mer_score_max,mer_hauteur_min,mer_hauteur_max,temps_precipitation,...,zone_SAINTE-MARIE A MAHANORO,zone_SAINTE-MARIE A MANANJARY,zone_SAINTE-MARIE A TAOLAGNARO,zone_SAINTE-MARIE AU CAP SAINTE-MARIE,zone_TAOLAGNARO AU CAP SAINTE MARIE,zone_TOAMASINA A TAOLAGNARO,zone_TOAMASINA AU CAP SAINTE-MARIE,zone_TOAMASINA AU TAOLAGNARO,zone_TOLIARA A TAOLAGNARO,zone_TOLIARA AU CAP SAINTE-MARIE
0,2019-09-06,10.0,15.0,25.0,90.0,5.0,6.0,1.25,4.0,1,...,False,False,False,False,False,False,False,False,False,False
1,2019-09-06,5.0,10.0,30.0,90.0,5.0,6.0,1.25,4.0,1,...,False,False,False,False,False,False,False,False,False,False
2,2019-09-06,15.0,20.0,20.0,90.0,NaN,NaN,NaN,NaN,0,...,False,False,False,False,False,False,False,False,False,False
3,2019-09-06,15.0,20.0,NaN,NaN,5.0,7.0,1.25,6.0,0,...,False,False,False,False,False,False,False,False,False,False
4,2019-09-06,20.0,25.0,35.0,90.0,NaN,NaN,NaN,NaN,0,...,False,False,False,False,False,False,False,False,False,False


---
## 8. Gestion des Valeurs Manquantes

In [12]:
print(" GESTION DES VALEURS MANQUANTES")
print("=" * 60)

# Vérifier les valeurs manquantes
missing = df_final.isnull().sum()
missing_cols = missing[missing > 0]

if len(missing_cols) > 0:
    print("\nColonnes avec valeurs manquantes:")
    for col, count in missing_cols.items():
        print(f"  {col}: {count} ({100*count/len(df_final):.1f}%)")
    
    # Imputer par la médiane pour les colonnes numériques
    for col in missing_cols.index:
        if df_final[col].dtype in ['float64', 'int64']:
            median_val = df_final[col].median()
            df_final[col].fillna(median_val, inplace=True)
            print(f"  → {col} imputé avec médiane: {median_val:.2f}")
else:
    print("\n Aucune valeur manquante")

 GESTION DES VALEURS MANQUANTES

Colonnes avec valeurs manquantes:
  vent_vitesse_min: 642 (47.7%)
  vent_vitesse_max: 642 (47.7%)
  vent_rafales: 628 (46.7%)
  vent_direction_deg: 187 (13.9%)
  mer_score_min: 253 (18.8%)
  mer_score_max: 253 (18.8%)
  mer_hauteur_min: 253 (18.8%)
  mer_hauteur_max: 253 (18.8%)
  → vent_vitesse_min imputé avec médiane: 10.00
  → vent_vitesse_max imputé avec médiane: 15.00
  → vent_rafales imputé avec médiane: 20.00
  → vent_direction_deg imputé avec médiane: 90.00
  → mer_score_min imputé avec médiane: 4.00
  → mer_score_max imputé avec médiane: 5.00
  → mer_hauteur_min imputé avec médiane: 0.50
  → mer_hauteur_max imputé avec médiane: 2.50


---
## 9. Création des Features Dérivées

In [13]:
print(" CRÉATION DE FEATURES DÉRIVÉES ET LAGS")
print("=" * 60)

# 1. Features météo dérivées
if 'vent_vitesse_max' in df_final.columns and 'vent_vitesse_min' in df_final.columns:
    df_final['vent_amplitude'] = df_final['vent_vitesse_max'] - df_final['vent_vitesse_min']
    df_final['vent_moyen'] = (df_final['vent_vitesse_max'] + df_final['vent_vitesse_min']) / 2

if 'mer_hauteur_max' in df_final.columns and 'mer_hauteur_min' in df_final.columns:
    df_final['mer_amplitude'] = df_final['mer_hauteur_max'] - df_final['mer_hauteur_min']
    df_final['mer_moyenne'] = (df_final['mer_hauteur_max'] + df_final['mer_hauteur_min']) / 2

# Indicateurs de conditions dangereuses
if 'vent_vitesse_max' in df_final.columns:
    df_final['vent_fort'] = (df_final['vent_vitesse_max'] >= 25).astype(int)  # > 25 kt = vent fort
    df_final['vent_violent'] = (df_final['vent_vitesse_max'] >= 40).astype(int)  # > 40 kt = violent

if 'vent_rafales' in df_final.columns:
    df_final['rafales_fortes'] = (df_final['vent_rafales'] >= 30).astype(int)

if 'mer_score_max' in df_final.columns:
    df_final['mer_agitee'] = (df_final['mer_score_max'] >= 4).astype(int)  # Douglas >= 4
    df_final['mer_forte'] = (df_final['mer_score_max'] >= 6).astype(int)  # Douglas >= 6

# 2. Features de décalage (LAGS) - Important pour la prédiction
# Trier par zone et date avant de calculer les lags
df_final = df_final.sort_values(['date'])

# Colonnes pour lesquelles créer des lags
lag_columns = []
if 'vent_vitesse_max' in df_final.columns:
    lag_columns.append('vent_vitesse_max')
if 'mer_hauteur_max' in df_final.columns:
    lag_columns.append('mer_hauteur_max')
if 'score_risque' in df_final.columns:
    lag_columns.append('score_risque')
if 'vent_moyen' in df_final.columns:
    lag_columns.append('vent_moyen')

for col in lag_columns:
    # Lag 1 jour
    df_final[f'{col}_lag1'] = df_final[col].shift(1)
    # Tendance (variation par rapport à la veille)
    df_final[f'{col}_trend'] = df_final[col] - df_final[f'{col}_lag1']

print(f" Features dérivées créées:")
print(f"   • Amplitudes et moyennes")
print(f"   • Indicateurs de danger (vent fort, mer agitée, etc.)")
print(f"   • Lags temporels (t-1) pour {len(lag_columns)} variables")
print(f"   • Tendances (différence t - t-1)")
print(f"\n Dimensions avant nettoyage: {df_final.shape}")

# 3. GESTION DES VALEURS MANQUANTES (CRITIQUE!)
print(f"\n GESTION DES VALEURS MANQUANTES:")

# Compter les NaN par colonne
nan_counts = df_final.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]

if len(nan_cols) > 0:
    print(f"   Colonnes avec NaN: {len(nan_cols)}")
    for col, count in nan_cols.items():
        pct = 100 * count / len(df_final)
        print(f"     • {col}: {count} ({pct:.1f}%)")

    # STRATÉGIE: Imputer intelligemment au lieu de supprimer
    for col in nan_cols.index:
        if col.endswith('_lag1'):
            # Pour les lags du premier jour: utiliser la valeur actuelle (pas de changement)
            df_final[col].fillna(df_final[col.replace('_lag1', '')], inplace=True)
        elif col.endswith('_trend'):
            # Pour les tendances du premier jour: 0 (pas de tendance)
            df_final[col].fillna(0, inplace=True)
        else:
            # Pour les autres colonnes numériques: médiane
            if df_final[col].dtype in ['float64', 'int64']:
                median_val = df_final[col].median()
                df_final[col].fillna(median_val, inplace=True)
                print(f"     → {col} imputé avec médiane: {median_val:.2f}")

# Vérifier qu'il ne reste plus de NaN
remaining_nan = df_final.isnull().sum().sum()
print(f"\n NaN restants après imputation: {remaining_nan}")
print(f" Dimensions finales après imputation: {df_final.shape}")


 CRÉATION DE FEATURES DÉRIVÉES ET LAGS
 Features dérivées créées:
   • Amplitudes et moyennes
   • Indicateurs de danger (vent fort, mer agitée, etc.)
   • Lags temporels (t-1) pour 4 variables
   • Tendances (différence t - t-1)

 Dimensions avant nettoyage: (1346, 100)

 GESTION DES VALEURS MANQUANTES:
   Colonnes avec NaN: 8
     • vent_vitesse_max_lag1: 1 (0.1%)
     • vent_vitesse_max_trend: 1 (0.1%)
     • mer_hauteur_max_lag1: 1 (0.1%)
     • mer_hauteur_max_trend: 1 (0.1%)
     • score_risque_lag1: 1 (0.1%)
     • score_risque_trend: 1 (0.1%)
     • vent_moyen_lag1: 1 (0.1%)
     • vent_moyen_trend: 1 (0.1%)

 NaN restants après imputation: 0
 Dimensions finales après imputation: (1346, 100)


---
## 10. Split Train/Test et Export

In [14]:
from sklearn.model_selection import train_test_split

print(" PRÉPARATION DES DATASETS FINAUX")
print("=" * 60)

# Supprimer les lignes avec des NaN (dues aux lags)
df_final_clean = df_final.dropna().copy()
print(f"\n Nettoyage: {len(df_final)} → {len(df_final_clean)} lignes (après suppression des NaN)")

# Colonnes à exclure des features
cols_to_exclude = ['date', 'incident']

# Sélectionner les features
feature_cols = [c for c in df_final_clean.columns if c not in cols_to_exclude]
print(f"\n Features sélectionnées: {len(feature_cols)}")

# Préparer X et y
X = df_final_clean[feature_cols]
y = df_final_clean['incident']

print(f"\n ANALYSE DU DÉSÉQUILIBRE:")
print(f"   Total: {len(y)} observations")
print(f"   Incidents (1): {y.sum()} ({100*y.mean():.2f}%)")
print(f"   Non-incidents (0): {(y == 0).sum()} ({100*(1-y.mean()):.2f}%)")
print(f"   Ratio: {int((y == 0).sum() / max(1, y.sum()))}:1")

# Vérifier qu'on a assez d'incidents pour faire un split stratifié
if y.sum() >= 2:
    # Split stratifié (préserve les proportions)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"\n Split stratifié réalisé")
else:
    # Split simple si trop peu d'incidents
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(f"\n Split simple (pas assez d'incidents pour stratification)")

print(f"\n RÉPARTITION TRAIN/TEST:")
print(f"   TRAIN: {len(X_train)} échantillons")
print(f"      - Incidents: {y_train.sum()} ({100*y_train.mean():.2f}%)")
print(f"      - Non-incidents: {(y_train == 0).sum()}")
print(f"   TEST: {len(X_test)} échantillons")
print(f"      - Incidents: {y_test.sum()} ({100*y_test.mean():.2f}%)")
print(f"      - Non-incidents: {(y_test == 0).sum()}")


 PRÉPARATION DES DATASETS FINAUX

 Nettoyage: 1346 → 1346 lignes (après suppression des NaN)

 Features sélectionnées: 98

 ANALYSE DU DÉSÉQUILIBRE:
   Total: 1346 observations
   Incidents (1): 168 (12.48%)
   Non-incidents (0): 1178 (87.52%)
   Ratio: 7:1

 Split stratifié réalisé

 RÉPARTITION TRAIN/TEST:
   TRAIN: 1076 échantillons
      - Incidents: 134 (12.45%)
      - Non-incidents: 942
   TEST: 270 échantillons
      - Incidents: 34 (12.59%)
      - Non-incidents: 236


---
## 11. Export des Datasets

In [15]:
print(" EXPORT DES DATASETS")
print("=" * 60)

# Créer les DataFrames pour l'export
train_df = X_train.copy()
train_df['incident'] = y_train.values

test_df = X_test.copy()
test_df['incident'] = y_test.values

# Exporter
train_df.to_csv(os.path.join(FINAL_PATH, 'dataset_train.csv'), index=False)
test_df.to_csv(os.path.join(FINAL_PATH, 'dataset_test.csv'), index=False)

# Exporter aussi le dataset complet
df_final.to_csv(os.path.join(FINAL_PATH, 'dataset_entrainement.csv'), index=False)

print("\n Fichiers exportés:")
print(f"   • dataset_train.csv ({len(train_df)} lignes)")
print(f"   • dataset_test.csv ({len(test_df)} lignes)")
print(f"   • dataset_entrainement.csv ({len(df_final)} lignes)")

 EXPORT DES DATASETS

 Fichiers exportés:
   • dataset_train.csv (1076 lignes)
   • dataset_test.csv (270 lignes)
   • dataset_entrainement.csv (1346 lignes)


In [16]:
print("=" * 60)
print(" CRÉATION DU DATASET TERMINÉE!")
print("=" * 60)

# Calculer des statistiques finales
total_incidents_dataset = df_final_clean['incident'].sum()
total_incidents_source = len(df_incidents_overlap)
taux_capture = 100 * total_incidents_dataset / len(incidents_nationaux) if len(incidents_nationaux) > 0 else 0

print(f"""
 RÉSUMÉ FINAL:
   
    INCIDENTS:
      - Incidents totaux (source): {len(df_incidents)}
      - Incidents dans période commune: {len(df_incidents_overlap)}
      - Jours uniques avec incidents: {len(incidents_nationaux)}
      - Incidents capturés dans dataset: {total_incidents_dataset}
      - Taux de capture: {taux_capture:.1f}%
   
    DATASET:
      - Total observations: {len(df_final_clean)} (Date x Zone)
      - Features météo: {len([c for c in feature_cols if any(k in c for k in ['vent', 'mer', 'temps'])])}
      - Features temporelles: {len([c for c in feature_cols if any(k in c for k in ['mois', 'jour', 'annee', 'saison'])])}
      - Features zones: {len([c for c in feature_cols if c.startswith('zone_')])}
      - Features dérivées/lags: {len([c for c in feature_cols if 'lag' in c or 'trend' in c or 'amplitude' in c])}
      - Total features: {len(feature_cols)}
   
    DÉSÉQUILIBRE:
      - Jours avec incident: {total_incidents_dataset} ({100*df_final_clean['incident'].mean():.1f}%)
      - Jours sans incident: {(df_final_clean['incident']==0).sum()} ({100*(1-df_final_clean['incident'].mean()):.1f}%)
      - Ratio: {int((df_final_clean['incident']==0).sum() / max(1, total_incidents_dataset))}:1

 FICHIERS CRÉÉS:
   • dataset_train.csv - {len(train_df)} lignes ({y_train.sum()} incidents)
   • dataset_test.csv - {len(test_df)} lignes ({y_test.sum()} incidents)
   • dataset_entrainement.csv - {len(df_final_clean)} lignes (dataset complet)

 PROCHAINE ÉTAPE: 
   Exécuter model.ipynb pour entraîner les modèles de prédiction
   Note: Le déséquilibre sera géré avec SMOTE et class_weight
""")

# Afficher quelques statistiques sur les incidents capturés
if total_incidents_dataset > 0:
    print("\n Distribution des incidents capturés par année:")
    incidents_dates = df_final_clean[df_final_clean['incident'] == 1]['date']
    if len(incidents_dates) > 0:
        print(incidents_dates.dt.year.value_counts().sort_index())


 CRÉATION DU DATASET TERMINÉE!

 RÉSUMÉ FINAL:

    INCIDENTS:
      - Incidents totaux (source): 269
      - Incidents dans période commune: 189
      - Jours uniques avec incidents: 167
      - Incidents capturés dans dataset: 168
      - Taux de capture: 100.6%

    DATASET:
      - Total observations: 1346 (Date x Zone)
      - Features météo: 28
      - Features temporelles: 10
      - Features zones: 55
      - Features dérivées/lags: 10
      - Total features: 98

    DÉSÉQUILIBRE:
      - Jours avec incident: 168 (12.5%)
      - Jours sans incident: 1178 (87.5%)
      - Ratio: 7:1

 FICHIERS CRÉÉS:
   • dataset_train.csv - 1076 lignes (134 incidents)
   • dataset_test.csv - 270 lignes (34 incidents)
   • dataset_entrainement.csv - 1346 lignes (dataset complet)

 PROCHAINE ÉTAPE: 
   Exécuter model.ipynb pour entraîner les modèles de prédiction
   Note: Le déséquilibre sera géré avec SMOTE et class_weight


 Distribution des incidents capturés par année:
date
2019    47
2020    